# This code trains a LSTM to predict if the news are true or fake based on https://www.kaggle.com/datasets/saurabhshahane/fake-news-classification/code

## Victoria Knapp Perez and Jason LaRuez

In [ ]:
! pip install -Uqq fastbook

In [ ]:
import fastbook

# Importing data
import kagglehub
import os
import json


import matplotlib.pyplot as plt # Plotting
import pandas as pd # Pandas dataframes

from fastbook import *
from fastai.text.all import *
import torch


from fastai.callback.tracker import SaveModelCallback, EarlyStoppingCallback #early stopper
from fastai.callback.rnn import RNNRegularizer #Regulariza RNN
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix #Analytics tools



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

out_dir = '/content/drive/MyDrive/NLPforFakeNews'
os.makedirs(out_dir, exist_ok=True)

# Add this directory to Python's search path
sys.path.append(out_dir)
from PreProcessingDataset import PreprocessDataOneDF

Mounted at /content/drive


In [ ]:


# Download kaggle dataset from https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset/data
path = kagglehub.dataset_download("saurabhshahane/fake-news-classification")



100%|██████████| 92.1M/92.1M [00:00<00:00, 217MB/s]

Extracting files...


In [ ]:
files = os.listdir(path)
files #See which files are inside

['WELFake_Dataset.csv']

In [ ]:
news_file = os.path.join(path, "WELFake_Dataset.csv")

df_news = pd.read_csv(news_file) #Read csv file as pd




In [ ]:
# Clean dataset with PreprocessData
train_df, val_df, test_df = PreprocessDataOneDF(
    df_news, 0.7, 0.2, 0.1,
    tfidf_radius=0.05, dedup_within_label=True,
    post_cross_exact_dedup=True,
    post_cross_near_dedup=True, post_radius=0.03
)
print("Split sizes:", len(train_df), len(val_df), len(test_df))
print("Train label counts:\n", train_df['label'].value_counts().sort_index())
print("Val   label counts:\n",   val_df['label'].value_counts().sort_index())
print("Test  label counts:\n",  test_df['label'].value_counts().sort_index())


Split sizes: 43140 12325 6161
Train label counts:
 label
0    24010
1    19130
Name: count, dtype: int64
Val   label counts:
 label
0    6860
1    5465
Name: count, dtype: int64
Test  label counts:
 label
0    3428
1    2733
Name: count, dtype: int64


In [ ]:
#Save the datasets
proc_dir = os.path.join(out_dir, "processed_datasets")
os.makedirs(proc_dir, exist_ok=True)

# Define file paths inside that folder
train_path = os.path.join(proc_dir, "train_FvT.csv")
val_path   = os.path.join(proc_dir, "val_FvT.csv")
test_path  = os.path.join(proc_dir, "test_FvT.csv")


train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)


In [ ]:
# Read the datasets
train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

In [ ]:

# Combine training and validation dataset for dataloader and learner
df_combined = pd.concat([train_df, val_df], ignore_index=True)
valid_idx = range(len(train_df), len(df_combined))

# Training LLM encoder

In [ ]:

#Relabel columns
df_combined = df_combined.rename(columns={'text_full':'text'})
df_combined['text']  = df_combined['text'].astype(str)
df_combined['label'] = df_combined['label'].astype(str).str.strip()


# 1) Text block for LM
tb_lm = TextBlock.from_df(
    'text',
    is_lm=True,
    seq_len=80
)

# 2) DataBlock with explicit IndexSplitter
dblock_lm = DataBlock(
    blocks=tb_lm,                      # only X for LM; Y is auto-shifted
    get_x=ColReader('text'),
    splitter=IndexSplitter(valid_idx)
)

# 3) Build LM DataLoaders
dls_lm = dblock_lm.dataloaders(
    df_combined,
    bs=128
)


In [ ]:
cbs = [
    RNNRegularizer(alpha=2., beta=1.),
    GradientClip(1.0),
    SaveModelCallback(monitor='valid_loss', fname='finetuned', with_opt=False),
    EarlyStoppingCallback(monitor='valid_loss', patience=1),
] #Early stopper

learn = language_model_learner(
    dls_lm, AWD_LSTM,
    drop_mult=0.7, metrics=[accuracy, Perplexity()], wd=0.1, cbs=cbs,
    pretrained=True  # <- keep pretrained
).to_fp16()  #Learner

learn.path = Path(out_dir) / 'LSTM_parameters'
learn.path.mkdir(parents=True, exist_ok=True)  # make sure it exists
learn.model_dir = '.'

# training schedule
learn.unfreeze()
learn.fit_one_cycle(3, lr_max=slice(5e-5, 5e-3), wd=0.1) #3 training epochs

learn.load('finetuned') #Get best epoch from cbs

# Save model

learn.save_encoder('finetuned')


epoch,train_loss,valid_loss,accuracy,perplexity,time
0,3.871059,3.701945,0.343707,40.526054,13:34
1,3.771082,3.614940,0.354179,37.149113,13:13
2,3.707373,3.581125,0.357846,35.913906,13:15


Better model found at epoch 0 with valid_loss value: 3.7019450664520264.
Better model found at epoch 1 with valid_loss value: 3.6149399280548096.
Better model found at epoch 2 with valid_loss value: 3.5811245441436768.


# Text classifier

In [ ]:
dblock = DataBlock(
    blocks=(
        TextBlock.from_df(text_cols='text', seq_len=72, vocab=dls_lm.vocab),
        CategoryBlock()
    ),
    get_x=ColReader('text'),
    get_y=ColReader('label'),
    splitter=IndexSplitter(valid_idx)
) #Datablock for loading train and val sets

dls_clas = dblock.dataloaders(df_combined, bs=64) #Load train and val sets


cbs = [
    RNNRegularizer(alpha=2., beta=1.),
    GradientClip(1.0),
    SaveModelCallback(monitor='accuracy', fname='class', with_opt=False),
    EarlyStoppingCallback(monitor='accuracy', patience=0),
] #Early stopper

learn = text_classifier_learner(
    dls_clas, AWD_LSTM,
    drop_mult=0.5,
    metrics=accuracy,
    loss_func=CrossEntropyLossFlat()
) #Learner

# Load the finetuned encoder

learn.path = Path(out_dir) / 'LSTM_parameters'
learn.path.mkdir(parents=True, exist_ok=True)  # make sure it exists


learn.model_dir = '.'
learn.load_encoder('finetuned')

# Training schedule
learn.freeze()
learn.fit_one_cycle(1, 1e-3, wd=0.1, cbs=cbs)

learn.freeze_to(-2)
learn.fit_one_cycle(1, 5e-4, wd=0.1, cbs=cbs)

learn.unfreeze()
learn.fit_one_cycle(3, lr_max=slice(5e-5, 5e-3), wd=0.1, cbs=cbs)
learn.load('class')
learn.save('class')


epoch,train_loss,valid_loss,accuracy,time
0,0.088755,0.024785,0.991805,01:41


Better model found at epoch 0 with accuracy value: 0.9918052554130554.


epoch,train_loss,valid_loss,accuracy,time
0,0.132417,0.018822,0.993428,01:57


Better model found at epoch 0 with accuracy value: 0.9934279918670654.


epoch,train_loss,valid_loss,accuracy,time
0,0.049233,0.021793,0.995213,02:43
1,0.028830,0.028418,0.994807,02:43


Better model found at epoch 0 with accuracy value: 0.995212972164154.
No improvement since epoch 0: early stopping


Path('/content/drive/MyDrive/NLPforFakeNews/LSTM_parameters/class.pth')

In [ ]:
#Calculate test df accuracy
texts  = test_df['text_full'].astype(str).tolist()
test_dl = learn.dls.test_dl(texts)


y_idx = test_df['label']

preds = learn.get_preds(dl=test_dl)



from sklearn.metrics import accuracy_score

y_hat = preds[0].argmax(dim=1)

acc = accuracy_score(y_idx, y_hat)
print(f"\n Test accuracy. : {acc:.3f}\n")


 Test accuracy. : 0.996

